# Collection CRUD Walkthrough

This notebook demonstrates collection CRUD using the real repositories:
1. Create a workspace
2. Create a collection in that workspace
3. Fetch collection by ID
4. Update collection name
5. Delete collection
6. Verify deletion

Prerequisites:
- PostgreSQL running
- Migrations applied (`make migrate`)


In [ ]:
from pathlib import Path
from uuid import uuid4
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'playground' else cwd
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dacke.domain.aggregates.workspace import Workspace
from dacke.domain.aggregates.collection import Collection
from dacke.domain.values.collection import CollectionID
from dacke.infrastructure.config import AppSettings
from dacke.infrastructure.repositories.providers.postgres.repo_workspace import WorkspaceRepository
from dacke.infrastructure.repositories.providers.postgres.repo_collection import CollectionRepository


In [ ]:
settings = AppSettings()
workspace_repo = WorkspaceRepository(connection_string=settings.database_url)
collection_repo = CollectionRepository(connection_string=settings.database_url)

workspace = Workspace.create(f'notebook-workspace-{uuid4().hex[:8]}')
await workspace_repo.create(workspace)
print('Created workspace:', workspace.identity, workspace.name)

collection = Collection.create(
    name=f'notebook-collection-{uuid4().hex[:8]}',
    workspace_id=workspace.identity.value,
)
await collection_repo.save_collection(collection)
print('Created collection:', collection.identity, collection.name)


In [ ]:
fetched = await collection_repo.get_collection_by_id(
    CollectionID.from_hex(str(collection.identity))
)
print('Fetched collection:', fetched)


In [ ]:
fetched.update_name(f'notebook-collection-renamed-{uuid4().hex[:6]}')
await collection_repo.update_collection(fetched)
updated = await collection_repo.get_collection_by_id(
    CollectionID.from_hex(str(collection.identity))
)
print('Updated collection name:', updated.name)


In [ ]:
await collection_repo.delete_collection(CollectionID.from_hex(str(collection.identity)))
after_delete = await collection_repo.get_collection_by_id(
    CollectionID.from_hex(str(collection.identity))
)
print('After delete:', after_delete)


In [ ]:
# Cleanup workspace (optional)
saved_workspace = await workspace_repo.get_by_id(workspace.identity)
if saved_workspace is not None:
    await workspace_repo.delete(saved_workspace)

await collection_repo._disconnect()
await workspace_repo._disconnect()
print('Cleanup done')
